# JDsearch pipeline: results

Reads what the scripts wrote to `outputs/<scope>/`; trains nothing.

In [ ]:
import json
from pathlib import Path
import polars as pl
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCOPE = "full"                      # or "sample_2000" after the smoke test
OUT = ROOT / "outputs" / SCOPE
R, F = OUT / "results", OUT / "figures"
pl.Config.set_tbl_rows(60); pl.Config.set_tbl_cols(12); pl.Config.set_fmt_str_lengths(70)

## Test-set comparison of every model

In [ ]:
pl.read_csv(R / "summary_test.csv")

## History retention windows (interim feedback)

In [ ]:
display(Image(F / "history_timeline.png"))
eda = json.loads((R / "eda.json").read_text())
pl.DataFrame(eda["retention"]), eda["recency_confound"], eda["popularity_auc"]

## Two-tower training curves

In [ ]:
for f in sorted((OUT / "tower").glob("*_history.json")):
    h = pl.DataFrame(json.loads(f.read_text()))
    print(f.stem); display(h.select("epoch", "train_loss", "mrr", "ndcg10", "gndcg10"))

## Re-ranker: feature importance and SHAP

In [ ]:
display(pl.read_csv(R / "ranker_feature_importance.csv").head(25))
display(Image(F / "diag_shap.png"))
pl.read_csv(R / "diag_shap_groups.csv")

## Where the model helps: segments

In [ ]:
pl.read_csv(R / "diag_segments.csv")

## Ablations (if run) and tower attention

In [ ]:
if (R / "diag_ablations.csv").exists():
    display(pl.read_csv(R / "diag_ablations.csv"))
json.loads((R / "diag_attention.json").read_text())